In [1]:
import csv
import os
from datetime import datetime, timedelta


# ============================================================
# BOOK CLASS
# ============================================================

class Book:
    def __init__(self, book_id, title, author, category):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.category = category
        self.available = True

    def display(self):
        status = "Available" if self.available else "Issued"

        print(
            f"{self.book_id:<8}"
            f"{self.title:<30}"
            f"{self.author:<25}"
            f"{self.category:<15}"
            f"{status:<10}"
        )


# ============================================================
# MEMBER CLASS
# ============================================================

class Member:
    def __init__(self, member_id, name, department):
        self.member_id = member_id
        self.name = name
        self.department = department

    def display(self):
        print(
            f"{self.member_id:<10}"
            f"{self.name:<25}"
            f"{self.department:<20}"
        )


# ============================================================
# LIBRARY CLASS
# ============================================================

class Library:

    def __init__(self):
        self.books = []
        self.members = []
        self.issued_books = {}

        # Tuple - fixed fine information
        self.fine_details = (5, 50)   # ₹5/day, maximum ₹50

        # Set - stores unique categories
        self.categories = set()

        # Dictionary - statistics
        self.statistics = {
            "books_added": 0,
            "books_issued": 0,
            "books_returned": 0
        }

        self.load_data()

    # ========================================================
    # FILE HANDLING
    # ========================================================

    def load_data(self):

        # Load books
        if os.path.exists("books.csv"):
            try:
                with open("books.csv", "r", newline="") as file:
                    reader = csv.DictReader(file)

                    for row in reader:
                        book = Book(
                            row["book_id"],
                            row["title"],
                            row["author"],
                            row["category"]
                        )

                        book.available = row["available"] == "True"

                        self.books.append(book)
                        self.categories.add(row["category"])

            except Exception as e:
                print("Error loading books:", e)

        # Load members
        if os.path.exists("members.csv"):
            try:
                with open("members.csv", "r", newline="") as file:
                    reader = csv.DictReader(file)

                    for row in reader:
                        member = Member(
                            row["member_id"],
                            row["name"],
                            row["department"]
                        )

                        self.members.append(member)

            except Exception as e:
                print("Error loading members:", e)

        # Load issued books
        if os.path.exists("issued.csv"):
            try:
                with open("issued.csv", "r", newline="") as file:
                    reader = csv.DictReader(file)

                    for row in reader:
                        self.issued_books[row["book_id"]] = {
                            "member_id": row["member_id"],
                            "issue_date": row["issue_date"],
                            "due_date": row["due_date"]
                        }

            except Exception as e:
                print("Error loading issued books:", e)

    # ========================================================

    def save_data(self):

        # Save books
        with open("books.csv", "w", newline="") as file:

            fieldnames = [
                "book_id",
                "title",
                "author",
                "category",
                "available"
            ]

            writer = csv.DictWriter(
                file,
                fieldnames=fieldnames
            )

            writer.writeheader()

            for book in self.books:
                writer.writerow({
                    "book_id": book.book_id,
                    "title": book.title,
                    "author": book.author,
                    "category": book.category,
                    "available": book.available
                })

        # Save members
        with open("members.csv", "w", newline="") as file:

            fieldnames = [
                "member_id",
                "name",
                "department"
            ]

            writer = csv.DictWriter(
                file,
                fieldnames=fieldnames
            )

            writer.writeheader()

            for member in self.members:
                writer.writerow({
                    "member_id": member.member_id,
                    "name": member.name,
                    "department": member.department
                })

        # Save issued books
        with open("issued.csv", "w", newline="") as file:

            fieldnames = [
                "book_id",
                "member_id",
                "issue_date",
                "due_date"
            ]

            writer = csv.DictWriter(
                file,
                fieldnames=fieldnames
            )

            writer.writeheader()

            for book_id, details in self.issued_books.items():

                writer.writerow({
                    "book_id": book_id,
                    "member_id": details["member_id"],
                    "issue_date": details["issue_date"],
                    "due_date": details["due_date"]
                })

    # ========================================================
    # ADD BOOK
    # ========================================================

    def add_book(self):

        print("\n========== ADD BOOK ==========")

        book_id = input("Enter Book ID: ").strip()

        # Check duplicate ID
        for book in self.books:
            if book.book_id == book_id:
                print("Book ID already exists!")
                return

        title = input("Enter Book Title: ").strip()
        author = input("Enter Author Name: ").strip()
        category = input("Enter Category: ").strip()

        if not title or not author or not category:
            print("All fields are required!")
            return

        book = Book(
            book_id,
            title,
            author,
            category
        )

        self.books.append(book)

        self.categories.add(category)

        self.statistics["books_added"] += 1

        self.save_data()

        print("Book added successfully!")

    # ========================================================
    # DISPLAY BOOKS
    # ========================================================

    def display_books(self):

        print("\n================ ALL BOOKS ================")

        if len(self.books) == 0:
            print("No books available.")
            return

        print(
            f"{'ID':<8}"
            f"{'Title':<30}"
            f"{'Author':<25}"
            f"{'Category':<15}"
            f"{'Status':<10}"
        )

        print("-" * 88)

        for book in self.books:
            book.display()

    # ========================================================
    # SEARCH BOOK
    # ========================================================

    def search_book(self):

        print("\n========== SEARCH BOOK ==========")

        keyword = input(
            "Enter title, author or category: "
        ).strip().lower()

        found = False

        for book in self.books:

            if (
                keyword in book.title.lower()
                or keyword in book.author.lower()
                or keyword in book.category.lower()
            ):

                if not found:
                    print()
                    print(
                        f"{'ID':<8}"
                        f"{'Title':<30}"
                        f"{'Author':<25}"
                        f"{'Category':<15}"
                        f"{'Status':<10}"
                    )

                    print("-" * 88)

                book.display()
                found = True

        if not found:
            print("No matching book found.")

    # ========================================================
    # ADD MEMBER
    # ========================================================

    def add_member(self):

        print("\n========== REGISTER MEMBER ==========")

        member_id = input("Enter Member ID: ").strip()

        for member in self.members:
            if member.member_id == member_id:
                print("Member ID already exists!")
                return

        name = input("Enter Member Name: ").strip()
        department = input("Enter Department: ").strip()

        if not name or not department:
            print("All fields are required!")
            return

        member = Member(
            member_id,
            name,
            department
        )

        self.members.append(member)

        self.save_data()

        print("Member registered successfully!")

    # ========================================================
    # DISPLAY MEMBERS
    # ========================================================

    def display_members(self):

        print("\n============== MEMBERS ==============")

        if not self.members:
            print("No members registered.")
            return

        print(
            f"{'ID':<10}"
            f"{'Name':<25}"
            f"{'Department':<20}"
        )

        print("-" * 55)

        for member in self.members:
            member.display()

    # ========================================================
    # ISSUE BOOK
    # ========================================================

    def issue_book(self):

        print("\n========== ISSUE BOOK ==========")

        book_id = input("Enter Book ID: ").strip()
        member_id = input("Enter Member ID: ").strip()

        selected_book = None
        selected_member = None

        # Find book
        for book in self.books:
            if book.book_id == book_id:
                selected_book = book
                break

        # Find member
        for member in self.members:
            if member.member_id == member_id:
                selected_member = member
                break

        if selected_book is None:
            print("Book not found!")
            return

        if selected_member is None:
            print("Member not found!")
            return

        if not selected_book.available:
            print("Book is already issued!")
            return

        # Maximum 3 books per member
        count = 0

        for details in self.issued_books.values():
            if details["member_id"] == member_id:
                count += 1

        if count >= 3:
            print("A member cannot issue more than 3 books.")
            return

        issue_date = datetime.now()
        due_date = issue_date + timedelta(days=7)

        selected_book.available = False

        self.issued_books[book_id] = {
            "member_id": member_id,
            "issue_date": issue_date.strftime("%Y-%m-%d"),
            "due_date": due_date.strftime("%Y-%m-%d")
        }

        self.statistics["books_issued"] += 1

        self.save_data()

        print("\nBook issued successfully!")
        print("Issue Date :", issue_date.strftime("%Y-%m-%d"))
        print("Due Date   :", due_date.strftime("%Y-%m-%d"))

    # ========================================================
    # RETURN BOOK
    # ========================================================

    def return_book(self):

        print("\n========== RETURN BOOK ==========")

        book_id = input("Enter Book ID: ").strip()

        if book_id not in self.issued_books:
            print("This book is not currently issued.")
            return

        details = self.issued_books[book_id]

        due_date = datetime.strptime(
            details["due_date"],
            "%Y-%m-%d"
        )

        return_date = datetime.now()

        late_days = (return_date - due_date).days

        fine_per_day, maximum_fine = self.fine_details

        if late_days > 0:

            fine = late_days * fine_per_day

            if fine > maximum_fine:
                fine = maximum_fine

            print("Book returned late!")
            print("Late Days :", late_days)
            print("Fine      : ₹", fine)

        else:
            fine = 0
            print("Book returned on time.")
            print("Fine      : ₹0")

        # Make book available
        for book in self.books:

            if book.book_id == book_id:
                book.available = True
                break

        del self.issued_books[book_id]

        self.statistics["books_returned"] += 1

        self.save_data()

        print("Book returned successfully!")

    # ========================================================
    # ISSUED BOOKS
    # ========================================================

    def show_issued_books(self):

        print("\n========== ISSUED BOOKS ==========")

        if not self.issued_books:
            print("No books are currently issued.")
            return

        print(
            f"{'Book ID':<10}"
            f"{'Member ID':<12}"
            f"{'Issue Date':<15}"
            f"{'Due Date':<15}"
        )

        print("-" * 52)

        for book_id, details in self.issued_books.items():

            print(
                f"{book_id:<10}"
                f"{details['member_id']:<12}"
                f"{details['issue_date']:<15}"
                f"{details['due_date']:<15}"
            )

    # ========================================================
    # REMOVE BOOK
    # ========================================================

    def remove_book(self):

        print("\n========== REMOVE BOOK ==========")

        book_id = input("Enter Book ID: ").strip()

        for book in self.books:

            if book.book_id == book_id:

                if not book.available:
                    print("Cannot remove an issued book!")
                    return

                self.books.remove(book)

                self.save_data()

                print("Book removed successfully!")
                return

        print("Book not found.")

    # ========================================================
    # CATEGORIES
    # ========================================================

    def show_categories(self):

        print("\n========== BOOK CATEGORIES ==========")

        if not self.categories:
            print("No categories available.")
            return

        for category in sorted(self.categories):
            print("-", category)

    # ========================================================
    # STATISTICS
    # ========================================================

    def show_statistics(self):

        total_books = len(self.books)

        available_books = 0
        issued_books = len(self.issued_books)

        for book in self.books:
            if book.available:
                available_books += 1

        print("\n========== LIBRARY STATISTICS ==========")

        print("Total Books       :", total_books)
        print("Available Books   :", available_books)
        print("Issued Books      :", issued_books)
        print("Total Members     :", len(self.members))
        print("Categories        :", len(self.categories))

        print("\nSession Statistics")

        for key, value in self.statistics.items():
            print(key.replace("_", " ").title(), ":", value)


# ============================================================
# INPUT VALIDATION FUNCTION
# ============================================================

def get_choice():

    while True:

        try:
            choice = int(input("\nEnter your choice: "))

            if 1 <= choice <= 12:
                return choice

            print("Please enter a number between 1 and 12.")

        except ValueError:
            print("Invalid input! Please enter a number.")


# ============================================================
# MAIN MENU
# ============================================================

def main():

    library = Library()

    while True:

        print("\n")
        print("=" * 50)
        print("        LIBRARY MANAGEMENT SYSTEM")
        print("=" * 50)

        print("1.  Add Book")
        print("2.  Display All Books")
        print("3.  Search Book")
        print("4.  Register Member")
        print("5.  Display Members")
        print("6.  Issue Book")
        print("7.  Return Book")
        print("8.  View Issued Books")
        print("9.  Remove Book")
        print("10. View Categories")
        print("11. Library Statistics")
        print("12. Exit")

        choice = get_choice()

        if choice == 1:
            library.add_book()

        elif choice == 2:
            library.display_books()

        elif choice == 3:
            library.search_book()

        elif choice == 4:
            library.add_member()

        elif choice == 5:
            library.display_members()

        elif choice == 6:
            library.issue_book()

        elif choice == 7:
            library.return_book()

        elif choice == 8:
            library.show_issued_books()

        elif choice == 9:
            library.remove_book()

        elif choice == 10:
            library.show_categories()

        elif choice == 11:
            library.show_statistics()

        elif choice == 12:
            print("\nThank you for using Library Management System!")
            print("All data has been saved.")
            break


# ============================================================
# PROGRAM START
# ============================================================

if __name__ == "__main__":
    main()



        LIBRARY MANAGEMENT SYSTEM
1.  Add Book
2.  Display All Books
3.  Search Book
4.  Register Member
5.  Display Members
6.  Issue Book
7.  Return Book
8.  View Issued Books
9.  Remove Book
10. View Categories
11. Library Statistics
12. Exit

========== ADD BOOK ==========
Book added successfully!


        LIBRARY MANAGEMENT SYSTEM
1.  Add Book
2.  Display All Books
3.  Search Book
4.  Register Member
5.  Display Members
6.  Issue Book
7.  Return Book
8.  View Issued Books
9.  Remove Book
10. View Categories
11. Library Statistics
12. Exit

================ ALL BOOKS ================
ID      Title                         Author                   Category       Status    
----------------------------------------------------------------------------------------
123     the harry potter              rk rowlling              adventure      Available 


        LIBRARY MANAGEMENT SYSTEM
1.  Add Book
2.  Display All Books
3.  Search Book
4.  Register Member
5.  Display Members
6.  Issu